# XGBoost Pixel Classification on Kaggle

这个 notebook 使用像元级 `XGBoost` 做 LULC 分类，并完成：
- 平衡抽样训练
- 测试集评估
- 全图预测
- 与 `LULC` 栅格逐像元对比验证精度

输入特征：`RGB + B8 + B11 + B12 + DEM`

标签策略：
- 忽略 `0、20、90`
- 保留 `10、30、40、50、60、80`
- 映射为 `0~5`


## 1. 导入包

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
import joblib

np.random.seed(42)

## 2. Kaggle 路径设置

In [ ]:
dataset_dir = Path('/kaggle/input/datasets/jasonwang2005/guanting/RS')
work_dir = Path('/kaggle/working/xgb_out')
work_dir.mkdir(parents=True, exist_ok=True)

rgb_path = dataset_dir / 'Guanting_RGB_2021.tif'
s2_path = dataset_dir / 'Guanting_S2_B8_B11_B12_20m.tif'
dem_path = dataset_dir / 'Guanting_DEM_30m.tif'
lulc_path = dataset_dir / 'Guanting_LULC_2021.tif'

print(rgb_path)
print(s2_path)
print(dem_path)
print(lulc_path)

## 3. 参数设置

In [ ]:
samples_per_class = 100000
ignore_labels = {0, 20, 90}
keep_labels = [10, 30, 40, 50, 60, 80]
label_to_class = {label: i for i, label in enumerate(keep_labels)}
class_to_label = {i: label for label, i in label_to_class.items()}
num_classes = len(keep_labels)

samples_per_class, num_classes

## 4. 读取 RGB 和 LULC，建立参考网格

In [ ]:
with rasterio.open(rgb_path) as rgb_ds, rasterio.open(lulc_path) as lulc_ds:
    rgb = rgb_ds.read().astype(np.float32)
    lulc = lulc_ds.read(1)
    ref_profile = rgb_ds.profile.copy()
    ref_crs = rgb_ds.crs
    ref_transform = rgb_ds.transform
    ref_height = rgb_ds.height
    ref_width = rgb_ds.width

rgb = np.transpose(rgb, (1, 2, 0))
print('rgb:', rgb.shape)
print('lulc:', lulc.shape)

## 5. 重采样 20m 和 30m 数据

In [ ]:
def read_and_resample_multiband(src_path, ref_crs, ref_transform, ref_height, ref_width, resampling=Resampling.bilinear):
    with rasterio.open(src_path) as src:
        out = np.empty((src.count, ref_height, ref_width), dtype=np.float32)
        for i in range(1, src.count + 1):
            reproject(
                source=src.read(i).astype(np.float32),
                destination=out[i - 1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=resampling
            )
        return np.transpose(out, (1, 2, 0))

def read_and_resample_singleband(src_path, ref_crs, ref_transform, ref_height, ref_width, resampling=Resampling.bilinear):
    with rasterio.open(src_path) as src:
        out = np.empty((ref_height, ref_width), dtype=np.float32)
        reproject(
            source=src.read(1).astype(np.float32),
            destination=out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=resampling
        )
        return out

s2_extra = read_and_resample_multiband(s2_path, ref_crs, ref_transform, ref_height, ref_width)
dem = read_and_resample_singleband(dem_path, ref_crs, ref_transform, ref_height, ref_width)

print('s2_extra:', s2_extra.shape)
print('dem:', dem.shape)

## 6. 归一化并堆叠特征

In [ ]:
def normalize_by_percentile(arr, low=2, high=98):
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    p_low = np.nanpercentile(arr, low)
    p_high = np.nanpercentile(arr, high)
    if p_high <= p_low:
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - p_low) / (p_high - p_low + 1e-6), 0, 1).astype(np.float32)

rgb_norm = normalize_by_percentile(rgb)
s2_norm = normalize_by_percentile(s2_extra)
dem_norm = normalize_by_percentile(dem)

X_full = np.dstack([rgb_norm, s2_norm, dem_norm[..., None]]).astype(np.float32)
print('X_full:', X_full.shape)

## 7. 构建有效像元并平衡抽样

In [ ]:
valid_mask = np.ones_like(lulc, dtype=bool)
for v in ignore_labels:
    valid_mask &= (lulc != v)

rows, cols = np.where(valid_mask)
labels_raw = lulc[rows, cols]

selected_idx = []
for raw_label in keep_labels:
    idx = np.where(labels_raw == raw_label)[0]
    n_take = min(samples_per_class, len(idx))
    chosen = np.random.choice(idx, size=n_take, replace=False)
    selected_idx.append(chosen)

selected_idx = np.concatenate(selected_idx)
selected_idx = np.random.permutation(selected_idx)

rows_sel = rows[selected_idx]
cols_sel = cols[selected_idx]
labels_sel_raw = labels_raw[selected_idx]
y = np.array([label_to_class[v] for v in labels_sel_raw], dtype=np.int64)
X = X_full[rows_sel, cols_sel, :]

print('X:', X.shape)
print('y:', y.shape)
for raw_label in keep_labels:
    print(raw_label, int(np.sum(labels_sel_raw == raw_label)))

## 8. 抽样可视化

In [ ]:
counts = [int(np.sum(labels_sel_raw == raw_label)) for raw_label in keep_labels]
plt.figure(figsize=(8, 4))
plt.bar([str(v) for v in keep_labels], counts)
plt.title('Balanced Samples Per Class')
plt.xlabel('LULC label')
plt.ylabel('sample count')
plt.show()

## 9. 划分训练、验证、测试集

In [ ]:
n = len(y)
idx = np.random.permutation(n)

n_train = int(n * 0.7)
n_val = int(n * 0.15)

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train+n_val]
test_idx = idx[n_train+n_val:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print('train:', X_train.shape, y_train.shape)
print('val:', X_val.shape, y_val.shape)
print('test:', X_test.shape, y_test.shape)

## 10. 训练 XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=num_classes,
    tree_method='hist',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50
)

joblib.dump(model, work_dir / 'xgb_model.joblib')
print(work_dir / 'xgb_model.joblib')

## 11. 绘制训练曲线

In [ ]:
results = model.evals_result()
epochs_x = range(len(results['validation_0']['mlogloss']))

plt.figure(figsize=(8, 4))
plt.plot(epochs_x, results['validation_0']['mlogloss'], label='train_mlogloss')
plt.plot(epochs_x, results['validation_1']['mlogloss'], label='val_mlogloss')
plt.xlabel('Iteration')
plt.ylabel('mlogloss')
plt.title('XGBoost Training Curve')
plt.legend()
plt.show()

## 12. 测试集评估

In [ ]:
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print('test_accuracy:', test_acc)
print(classification_report(y_test, y_pred, target_names=[str(v) for v in keep_labels], digits=4))

## 13. 混淆矩阵

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[str(v) for v in keep_labels])
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
plt.tight_layout()
plt.show()

## 14. 全图预测

In [ ]:
X_pred = X_full.reshape(-1, X_full.shape[-1])
pred_full_class = model.predict(X_pred).reshape(ref_height, ref_width)
pred_full_label = np.vectorize(class_to_label.get)(pred_full_class).astype(np.uint8)

print('pred_full_class:', pred_full_class.shape)
print('pred_full_label:', pred_full_label.shape)

## 15. 与 LULC 真值逐像元对比

In [ ]:
eval_mask = np.ones_like(lulc, dtype=bool)
for v in ignore_labels:
    eval_mask &= (lulc != v)

y_true_full = np.array([label_to_class[v] for v in lulc[eval_mask]], dtype=np.int64)
y_pred_full = pred_full_class[eval_mask]

full_acc = accuracy_score(y_true_full, y_pred_full)
print('full_raster_accuracy:', full_acc)
print(classification_report(y_true_full, y_pred_full, target_names=[str(v) for v in keep_labels], digits=4))

## 16. 保存预测栅格

In [ ]:
pred_path = work_dir / 'xgb_pred_lulc.tif'
profile = ref_profile.copy()
profile.update(dtype=rasterio.uint8, count=1, compress='lzw')

with rasterio.open(pred_path, 'w', **profile) as dst:
    dst.write(pred_full_label, 1)

print(pred_path)

## 17. 简单可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb_norm)
axes[0].set_title('RGB')
axes[1].imshow(lulc, interpolation='nearest')
axes[1].set_title('Reference LULC')
axes[2].imshow(pred_full_label, interpolation='nearest')
axes[2].set_title('XGBoost Prediction')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()